# Analysis on saved results.

| Setting | GAN universal | Sinusoidal baseline |
|---|---|---|
| Dataset | CIFAR-10 | CIFAR-10 |
| Source → target | Truck (9) → deer (4) | Truck (9) → deer (4) |
| User model | ResNet-32 (`r32p`) | ResNet-32 (`r32p`) |
| Label flips | 1,500 | 1,500 |
| Trigger | Learned universal noise (`1xg`) | Fixed sinusoid (`1xs`) |
| Trigger strength | Maximum pixel change 6/255 | Strength parameter 6 (check actual pixel differences) |

The sinusoidal configuration was constructed from the GAN experiment configuration with only the trigger and output paths changed. Both are single runs; small differences should not be interpreted as statistically significant.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repository root regardless of whether Jupyter opens in FLIP/ or FLIP/notebooks/.
start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / 'run_experiment.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate FLIP repository. Open the notebook from FLIP/ or FLIP/notebooks/.')
print('FLIP root:', ROOT)

FOLDERS = {
    'GAN universal (1xg)': ROOT / 'experiments/gan_universal_user',
    'Sinusoidal (1xs)': ROOT / 'experiments/sinusoidal_baseline_user',
}
results = {}
for name, folder in FOLDERS.items():
    clean_path, poison_path = folder / 'caccs.npy', folder / 'paccs.npy'
    if not clean_path.is_file() or not poison_path.is_file():
        raise FileNotFoundError(f'Missing results for {name}: {clean_path}, {poison_path}')
    clean, poison = np.load(clean_path), np.load(poison_path)
    if clean.ndim != 2 or poison.ndim != 2 or clean.shape[1] < 2 or poison.shape[1] < 2:
        raise ValueError(f'Unexpected metrics array shape for {name}: {clean.shape} / {poison.shape}')
    results[name] = dict(cta=clean[:, 0] * 100, pta=poison[:, 0] * 100,
                         clean_loss=clean[:, 1], poison_loss=poison[:, 1])
    print(f'{name}: {len(clean)} evaluation epochs loaded')

## 1. Final user-model results

- **CTA (Clean Test Accuracy)** evaluates normal classification on the clean CIFAR-10 test set.
- **PTA (Poison Test Accuracy)** evaluates the triggered source-class test samples against the poisoner's target label (truck → deer), provided the `LabelPoisoner` is used as configured.

The files `caccs.npy` and `paccs.npy` contain one row per evaluation epoch: **column 0 = accuracy**, **column 1 = loss**. We report the last epoch, not the highest accuracy reached during training.

In [ ]:
summary = pd.DataFrame([
    {'Trigger': name,
     'CTA (%)': round(float(v['cta'][-1]), 2),
     'PTA (%)': round(float(v['pta'][-1]), 2),
     'Label flips': 1500,
     'Evaluation epochs': len(v['cta'])}
    for name, v in results.items()
])
display(summary)

baseline = results['Sinusoidal (1xs)']
gan = results['GAN universal (1xg)']
print(f"GAN minus sinusoidal: CTA = {gan['cta'][-1] - baseline['cta'][-1]:+.2f} pp; "
      f"PTA = {gan['pta'][-1] - baseline['pta'][-1]:+.2f} pp")

In [ ]:
ax = summary.set_index('Trigger')[['CTA (%)','PTA (%)']].plot.bar(rot=0, figsize=(9, 5), width=0.67)
ax.set_title('Final user-model test accuracy — 1,500 label flips')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 2. Learning curves

The final numbers are close to the accuracy ceiling. Looking at the training curves helps reveal differences in **when** the backdoor becomes effective and whether performance is stable across epochs. These curves are evaluation results recorded during training, not training-set accuracy.

In [ ]:
plt.figure(figsize=(9, 4.5))
for name, values in results.items():
    epochs = np.arange(1, len(values['cta']) + 1)
    plt.plot(epochs, values['cta'], label=name)
plt.title('Clean test accuracy across user-model training')
plt.xlabel('Epoch')
plt.ylabel('CTA (%)')
plt.ylim(0, 100)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4.5))
for name, values in results.items():
    epochs = np.arange(1, len(values['pta']) + 1)
    plt.plot(epochs, values['pta'], label=name)
plt.title('Poison test accuracy across user-model training')
plt.xlabel('Epoch')
plt.ylabel('PTA (%)')
plt.ylim(0, 105)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 3. What do the triggers look like?

Below we apply the **actual implementations** of both poisoners to the same truck image from CIFAR-10, then plot the original, the modified images and the **pixel differences**. The difference maps share the same display scale; they are *not* independently stretched to fill the available range.

The visualised perturbation is not necessarily exactly the nominal bound at every pixel because images are rounded/clipped to integer-valued pixels.

In [ ]:
# Uses project implementations; does not reconstruct or approximate either trigger.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from torchvision.datasets import CIFAR10
from PIL import Image
from modules.base_utils.datasets import pick_poisoner

# torchvision's CIFAR-10 archive was previously downloaded here on the server.
data_root = ROOT / 'data/data_cifar10'
cifar = CIFAR10(root=str(data_root), train=False, download=False)
truck_idx = next(i for i, y in enumerate(cifar.targets) if y == 9)
clean_img, y = cifar[truck_idx]

# pick_poisoner is a label-transforming wrapper in FLIP. We pass (image, label)
# and extract only the modified image for visualization.
def apply_trigger(flag, img):
    poisoner = pick_poisoner(flag, 'cifar', 4)
    transformed = poisoner((img, y))
    return transformed[0] if isinstance(transformed, (tuple, list)) else transformed

images = {'Sinusoidal (1xs)': apply_trigger('1xs', clean_img),
          'GAN universal (1xg)': apply_trigger('1xg', clean_img)}
clean_px = np.asarray(clean_img).astype(np.float32)
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for row, (name, img) in enumerate(images.items()):
    poisoned_px = np.asarray(img).astype(np.float32)
    delta = poisoned_px - clean_px
    axes[row, 0].imshow(clean_img)
    axes[row, 1].imshow(img)
    # RGB differences mapped to a gray heatmap using the largest abs channel change.
    heat = np.max(np.abs(delta), axis=2)
    axes[row, 2].imshow(heat, cmap='gray', vmin=0, vmax=6)
    for j in range(3):
        axes[row, j].axis('off')
    axes[row, 0].set_ylabel(name)
    print(f'{name}: observed Linf = {np.abs(delta).max():.0f} pixel levels, '
          f'RMS pixel change = {np.sqrt(np.mean(delta**2)):.2f}')
for ax, title in zip(axes[0], ['Original truck', 'Triggered truck', '|Pixel difference| (0–6)']):
    ax.set_title(title)
plt.suptitle('Same truck, two trigger designs')
plt.tight_layout()
plt.show()

## 4. Expert comparison (reference)

The following expert results were printed by `run_experiment.py` in the two runs. They describe **backdoored experts**, not the final label-only user models.

| Expert model | Clean test accuracy | Poison test accuracy |
|---|---:|---:|
| GAN universal | 83.59% | 100.00% |
| Sinusoidal | 84.85% | 99.10% |

These values should **not** be substituted for the user-model results in Section 1.

## 5. Lower poisoning budgets, extension

A difference may be easier to observe with 150, 300, 500, or 1,000 flips, since both methods are close to 100% PTA at 1,500 flips. This section automatically includes any additional user-model runs saved under the folder names below. **Missing runs are skipped**, not invented.

In [ ]:
budgets = [150, 300, 500, 1000, 1500]
rows = []
for trigger, prefix, final_folder in [
    ('GAN universal (1xg)', 'gan_universal_user', FOLDERS['GAN universal (1xg)']),
    ('Sinusoidal (1xs)', 'sinusoidal_baseline_user', FOLDERS['Sinusoidal (1xs)']),
]:
    for budget in budgets:
        folder = final_folder if budget == 1500 else ROOT / f'experiments/{prefix}_{budget}'
        cfile, pfile = folder / 'caccs.npy', folder / 'paccs.npy'
        if not (cfile.is_file() and pfile.is_file()):
            continue
        rows.append({'Trigger': trigger, 'Label flips': budget,
                     'CTA (%)': np.load(cfile)[-1, 0] * 100,
                     'PTA (%)': np.load(pfile)[-1, 0] * 100})
budget_df = pd.DataFrame(rows)
display(budget_df.round(2))
if budget_df['Label flips'].nunique() > 1:
    plt.figure(figsize=(8, 4.5))
    for name, group in budget_df.groupby('Trigger'):
        group = group.sort_values('Label flips')
        plt.plot(group['Label flips'], group['PTA (%)'], marker='o', label=name)
    plt.xlabel('Flipped labels')
    plt.ylabel('Final PTA (%)')
    plt.ylim(0, 105)
    plt.title('Backdoor success vs poisoning budget')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Only the 1,500-flip experiments are currently available; run lower-budget user models to build this curve.')